In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import mean_squared_error, accuracy_score

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [54]:
# Feature Engineering
def feature_engineering(df):
    # Ensure the input is a DataFrame
    if isinstance(df, np.ndarray):
        df = pd.DataFrame(df)
    
    if 'TotalBsmtSF' in df.columns and '1stFlrSF' in df.columns and '2ndFlrSF' in df.columns:
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    if 'FullBath' in df.columns and 'HalfBath' in df.columns and 'BsmtFullBath' in df.columns and 'BsmtHalfBath' in df.columns:
        df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    if 'OpenPorchSF' in df.columns and 'EnclosedPorch' in df.columns and '3SsnPorch' in df.columns and 'ScreenPorch' in df.columns:
        df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
    
    if 'GrLivArea' in df.columns:
        df['GrLivArea'] = np.log1p(df['GrLivArea'])
    if 'TotalSF' in df.columns:
        df['TotalSF'] = np.log1p(df['TotalSF'])
    
    label_enc_cols = ['MSZoning', 'Street', 'Alley']
    for col in label_enc_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
    
    return df

In [33]:
train = feature_engineering(train)
test = feature_engineering(test)

In [37]:
def preprocess_data(df):
    if isinstance(df, np.ndarray):
        df = pd.DataFrame(df)
        
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    
    categorical_cols = df.select_dtypes(include=[object]).columns
    for col in categorical_cols:
        if df[col].mode().empty:
            df[col] = df[col].fillna('Unknown')
        else:
            df[col] = df[col].fillna(df[col].mode().iloc[0])
    
    df = pd.get_dummies(df)
    
    return df

In [38]:
train = preprocess_data(train)
test = preprocess_data(test)

In [39]:
missing_cols = set(train.columns) - set(test.columns)
for c in missing_cols:
    test[c] = 0
test = test[train.columns.drop('SalePrice')]

C:\Users\KIIT\AppData\Local\Temp\ipykernel_4036\2630352240.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[c] = 0
C:\Users\KIIT\AppData\Local\Temp\ipykernel_4036\2630352240.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[c] = 0
C:\Users\KIIT\AppData\Local\Temp\ipykernel_4036\2630352240.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get

In [40]:
test = test.drop('Id', axis=1)

In [41]:
X = train.drop(['Id', 'SalePrice'], axis=1)
y = train['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
test = scaler.transform(test)

In [43]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_val)
print('Linear Regression RMSE:', np.sqrt(mean_squared_error(y_val, lr_preds)))

Linear Regression RMSE: 1.2617247623104902e+17


In [44]:
poly = PolynomialFeatures(degree=2)
X_poly_train = poly.fit_transform(X_train)
X_poly_val = poly.transform(X_val)

poly_lr = LinearRegression()
poly_lr.fit(X_poly_train, y_train)
poly_preds = poly_lr.predict(X_poly_val)
print('Polynomial Regression RMSE:', np.sqrt(mean_squared_error(y_val, poly_preds)))

Polynomial Regression RMSE: 35708.11401671584


In [45]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_preds = ridge.predict(X_val)
print('Ridge Regression RMSE:', np.sqrt(mean_squared_error(y_val, ridge_preds)))

Ridge Regression RMSE: 27946.010205584378


In [46]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
lasso_preds = lasso.predict(X_val)
print('Lasso Regression RMSE:', np.sqrt(mean_squared_error(y_val, lasso_preds)))

Lasso Regression RMSE: 27728.418237883365


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.775e+10, tolerance: 6.967e+08
  model = cd_fast.enet_coordinate_descent(


In [47]:
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train, y_train)
elastic_preds = elastic.predict(X_val)
print('ElasticNet Regression RMSE:', np.sqrt(mean_squared_error(y_val, elastic_preds)))

ElasticNet Regression RMSE: 28834.415476068087


In [48]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
log_reg_preds = log_reg.predict(X_val)
print('Logistic Regression Accuracy:', accuracy_score(y_val, log_reg_preds))

Logistic Regression Accuracy: 0.003424657534246575


In [49]:
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_preds = nb.predict(X_val)
print('Naive Bayes Accuracy:', accuracy_score(y_val, nb_preds))

Naive Bayes Accuracy: 0.0136986301369863


In [50]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
knn_preds = knn.predict(X_val)
print('k-Nearest Neighbors Accuracy:', accuracy_score(y_val, knn_preds))

k-Nearest Neighbors Accuracy: 0.0


In [51]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_val)
print('Decision Trees Accuracy:', accuracy_score(y_val, dt_preds))

Decision Trees Accuracy: 0.003424657534246575


In [52]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_val)
print('Random Forest Accuracy:', accuracy_score(y_val, rf_preds))

Random Forest Accuracy: 0.003424657534246575


In [53]:
svm = SVC()
svm.fit(X_train, y_train)
svm_preds = svm.predict(X_val)
print('Support Vector Machines Accuracy:', accuracy_score(y_val, svm_preds))

Support Vector Machines Accuracy: 0.010273972602739725
